In [1]:

import csv, json, shutil
from pathlib import Path
from collections import Counter, defaultdict

LAST_NAME = "Tapiculin"
STUDENT_ID = "TUPM-26-1932"
SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"          # put the Kaggle CSV files here (git-ignored)
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"
for d in (DATA_DIR, OUTPUT_DIR, BACKUP_DIR):
    d.mkdir(exist_ok=True)
print("Seed:", SEED_DIGIT, "| ID Sum:", ID_SUM, "| Name Length:", NAME_LENGTH)

# %% EXERCISE 1 - Web Server Access Log Analysis (weblog.csv)
log_path = DATA_DIR / "weblog.csv"
RECORDS_TO_PROCESS = 1000 + ID_SUM
TARGET_CLASS = (SEED_DIGIT % 4) + 2                # 2 -> 2xx, 3 -> 3xx, 4 -> 4xx, 5 -> 5xx

selected, status_counts, url_counts = [], Counter(), Counter()
processed = skipped = 0
with open(log_path, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)                         # reads one line at a time
    header = [h.strip() for h in next(reader)]
    idx = {name: header.index(name) for name in ("IP", "Time", "URL")}
    status_col = "Staus" if "Staus" in header else "Status"   # dataset column is spelled 'Staus'
    idx["Status"] = header.index(status_col)
    for row in reader:
        if processed + skipped >= RECORDS_TO_PROCESS:
            break
        try:
            ip, ts, url = row[idx["IP"]], row[idx["Time"]], row[idx["URL"]]
            status = int(row[idx["Status"]])
        except (IndexError, ValueError):           # invalid or missing record
            skipped += 1
            continue
        processed += 1
        if status // 100 == TARGET_CLASS:
            selected.append((ip, ts, url, status))
            status_counts[status] += 1
            url_counts[url] += 1

top5 = url_counts.most_common(5)
filtered_csv = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_web_filtered.csv"
with open(filtered_csv, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f); w.writerow(["IP", "Time", "URL", "Status"]); w.writerows(selected)

json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_web_summary.json"
summary = {"records_processed": processed, "records_skipped": skipped, "target_status_class": f"{TARGET_CLASS}xx",
           "records_in_target_class": len(selected), "count_per_status": dict(status_counts),
           "five_most_requested_urls": top5}
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)
backup_path = BACKUP_DIR / json_path.name
shutil.copy2(json_path, backup_path)

print("Records processed:", processed, "(skipped invalid:", skipped, ")")
print("Target status class:", f"{TARGET_CLASS}xx")
print("Records in target class:", len(selected))
print("Count per status code:", dict(status_counts))
print("Five most requested URLs:", top5)
print("Filtered CSV file:", filtered_csv)
print("JSON summary file:", json_path)
print("Backup verification:", backup_path.exists())



Seed: 2 | ID Sum: 23 | Name Length: 9
Records processed: 1023 (skipped invalid: 0 )
Target status class: 4xx
Records in target class: 5
Count per status code: {404: 5}
Five most requested URLs: [('GET /robots.txt HTTP/1.1', 4), ('GET /favicon.ico HTTP/1.1', 1)]
Filtered CSV file: output\Tapiculin_TUPM-26-1932_web_filtered.csv
JSON summary file: output\Tapiculin_TUPM-26-1932_web_summary.json
Backup verification: True
